# CIII × TVB-20 — Demo Analysis

This notebook walks through the diagnostic-index and statistics pipeline using **synthetic** data.

**Important:** results shown here are illustrative. To run on real survey data, supply your own (anonymised) questionnaire file at the repo root as `questionnaire_responses_v21.csv` — see the main README for details.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from finance_math import future_value_monthly_payment, opportunity_cost, growth_curve
from ciii import score_ciii, classify_estimate, quadrant_split

## 1. Generate synthetic data

In [ ]:
import subprocess
subprocess.run(['python', 'generate_mock_data.py'], check=True)
df = pd.read_csv('data/synthetic_compound_intuition_responses.csv')
print(f'N = {len(df)}')
df.head()

## 2. Compound interest math demo

In [ ]:
oc = opportunity_cost(payment=300, annual_rate=0.08, years=18)
print(f"18-year principal:    MOP ${oc['principal']:>10,.0f}")
print(f"18-year compound FV:  MOP ${oc['future_value']:>10,.0f}")
print(f"Time-value added:      MOP ${oc['gap']:>10,.0f}")
print(f"Multiplier:            {oc['multiplier']:.2f}x")

In [ ]:
g = growth_curve(payment=300, annual_rate=0.08, years=18)
months_axis = [m / 12 for m in g['months']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(months_axis, g['principal'], label='Principal (linear)', linestyle=':', color='#999')
ax.plot(months_axis, g['compound'], label='Compound (8% annual)', color='#3498db', linewidth=2)
ax.fill_between(months_axis, g['principal'], g['compound'], alpha=0.2, color='#3498db',
                label='Time-value gap')
ax.set_xlabel('Years')
ax.set_ylabel('MOP')
ax.set_title('MOP 300/month, 8% annual — 18-year growth')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Diagnostic index (CIII)

In [ ]:
df['CIII'] = df.apply(lambda r: score_ciii(r['G1'], r['G2'], r['G3']), axis=1)
print(df['CIII'].describe())
print('\nCIII distribution:')
print(df['CIII'].value_counts().sort_index())

## 4. G3 estimation distribution

In [ ]:
def classify_g3(s):
    if s == '$120,000–$200,000（高回報）':
        return 'strict_correct'
    if s == '$60,000–$120,000（中等回報）':
        return 'near_correct'
    if s == '不清楚':
        return 'dont_know'
    return 'serious_low'

counts = df['G3'].apply(classify_g3).value_counts()
print('G3 classification:')
print(counts)
print(f"\nStrict correct: {counts.get('strict_correct', 0)} / {len(df)} = {counts.get('strict_correct', 0) / len(df) * 100:.1f}%")
print(f"Don't know:     {counts.get('dont_know', 0)} / {len(df)} = {counts.get('dont_know', 0) / len(df) * 100:.1f}%")

## 5. Quadrant split

In [ ]:
quad = quadrant_split(df['CIII'].tolist(), df['G9'].tolist())
print('Four-quadrant analysis:')
for k, v in quad.items():
    print(f'  {k}: {v} ({v / len(df) * 100:.1f}%)')

## 6. Run full pipeline

In [ ]:
subprocess.run(['python', 'analyze_from_csv.py'], check=True)